# The basics of NIM

## Introduction

![build.nvidia.com](images/build_nvidia.png)


In this notebook, we will do a hands on walkthrough of the NVIDIA NIM for LLMs to showcase how to get LLMs to production in a streamline fashion. The goal of the microservice is to simplify the process of building an optimized inference engine and deploying it to a production inference server. 

We will be leveraging models hosted on [build.nvidia.com](https://build.nvidia.com) to test a Mistral-7b NIM using cloud endpoints.  We will then deploy the model to a local NIM server (this launchpad instance) and test the model using a NIM client.

**Notebook Requirements**:
* Access to a single NVIDIA GPU (A30, A100, H100, L40S, A40)
* Access to build.nvidia.com - After the lab, sign up for access to the NVIDIA NGC and build.nvidia.com to access the models used in this lab.
* Docker
* Notebook is designed to be run from host system, and not within docker container.

In [ ]:
# Stop all prior running containers 
!scripts/stop_containers.sh

### Import Environment

In [ ]:
# Setup Environment
import os
from dotenv import load_dotenv
# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"Current working dir           : { os.getcwd() }")
print(f"NGC_API_KEY for lab use only  : {os.getenv('NGC_API_KEY')}")
print(f"NVIDIA_API_KEY for lab use only  : {os.getenv('NVIDIA_API_KEY')}")
LOCAL_MODEL_URL = "http://0.0.0.0:8000"



In [ ]:
# Check GPU availability
!nvidia-smi

### [Information Only - Nim Client - Instructions to build your own docker client ] 

In this lab, the environment is already setup for you.  
If you wanted to build a nim_client for yourself, here is a sample [nim client dockerfile](./docker/nim_client.Dockerfile)

We can build and run this with the following commands:

```bash
cd docker
# Build and run the docker container
docker compose up nim_client
```

As this notebook is running in this NVIDIA launchpad environment, we don't need to run the above command. We include it for posterity and for those who wish to download this notebook and run it on their own system (e.g. A100, H100, L40S).

### Rapid application experimentation using NVIDIA Foundation Endpoint build.nvidia.com

Many of our NIM LLM models support the OpenAI API for completion and chat-completion api calls.  
For more more information regarding the text generation models see [OpenAI documentation](https://platform.openai.com/docs/guides/text-generation)
We can use the OpenAI Chat Completion API and interact with our Llama 8B model.

Here we run inference on our NVIDIA cloud hosted models.  For more information, see [build.nvidia.com](https://build.nvidia.com/explore/discover).  If you sign up for an build.nvidia.com account, you can get cloud credits to run inference on a number of models.  This list is updated frequently and ever growing.

Below we'll try a simple API interaction with Mixtral 8x7B.

In [ ]:
messages = [
  {"role": "user", 
   "content": "You are a helpful assistant that provides succinct answers. What are the main branches of the Department of Defense?"}]

# Since NVIDIA NIM for LLMs offers an OpenAI-compatible API,
# we can use the OpenAI Python client
from openai import OpenAI


client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.environ["NGC_API_KEY"]
)


# Feel free to swap in some of these models !

# nvdev/meta/llama-3.1-70b-instruct
# nvdev/microsoft/phi-3-mini-4k-instruct
# nvdev/mistralai/mistral-7b-instruct-v0.3
# nvdev/mistralai/mixtral-8x22b-instruct-v0.1
# nvdev/meta/llama-3.2-3b-instruct

# Try comparing the response of the above model with a reasoning model!
# Currently we have following NIMs avaliable for reasoning models:
# nvdev/llama-3.3-nemotron-super-49b-v1
# nvdev/deepseek-ai/deepseek-r1

# Uncomment the following line to use the llama-3.3-nemotron-super-49b-v1 model.
"""
cloud_model = "nvdev/llama-3.3-nemotron-super-49b-v1"
# This will take several minutes since model does many inferences to get response.   
completion = client.chat.completions.create(
  model=cloud_model,
  messages=messages,
  temperature=0.6,
  top_p=0.7,
  max_tokens=1024,
  stream=True
)
"""

cloud_model = "nvdev/mistralai/mistral-7b-instruct-v0.3"

completion = client.chat.completions.create(
  model=cloud_model,
  messages=messages,
  temperature=0.5,
  top_p=1,
  max_tokens=1024,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

## Foundational concepts to use NVIDIA NIM for LLMs (Locally Hosted)

NIMs support two types of backends : VLLM and TensorRT-LLM.  
* VLLM : Open source inferencing package that is based solely on CUDA implementations and therefore can compile to wide range of architectures
* TensorRT-LLM : NVIDIAs library for running transformer models with the best performance possible.  Many kernel level optimizations.


Based on your hardware configuration, the type of model you get is automatically managed when a docker run command is launched.  The diagram below shows the details of the decision sequence.


<img src="images/pull_sequence2.png" alt="vllm trt seq" style="width:35%;">

## Additional Configuration Options
If you want to override the default configuration for NVIDIA NIMs, see this following [configuration guide](https://docs.nvidia.com/nim/large-language-models/latest/configuration.html#environment-variables)

You can control features such as:
* NIM_MODEL_PROFILE  : Override the default model profile
* NIM_LOG_LEVEL      : Set the log level for the NIM server
* NIM_CACHE_PATH     : location to hold the local model assets

## Using Prebuilt Models

There are two sites of interest for this lab

| Resource | Description | Link |
|----------|-------------|------|
| NVIDIA Inference Microservices (NIM) | The site for rapid application development with NIM endpoints. | [build.nvidia.com](https://build.nvidia.com/explore/discover) |
| NVIDIA GPU Cloud (NGC) | The site that warehouses the docker containers, yaml configurations and setup instructions to run the same functionality locally | [ngc.nvidia.com](https://ngc.nvidia.com) |



NVIDIA Inference Microservice (NIM) is bundled with prebuilt TRT-LLM engines for various models like Llama, Mistral, Mixtral, Gemma, or Phi3, as well as models for image generation, speech recognition, code generation and more. These prebuilt models can be used directly, and many can be fine-tuned for specialized performance on your use-case.

Keep checking [build.nvidia.com](build.nvidia.com) for the latest models and updates.

### Launch locally hosted Llama 3.1 8B Instruct model with NIM

Now lets launch the NVIDIA NIM for locally.


In this example, we specified to use GPU #0 on the system with `--gpus '"device=0"'`. If you don’t need to set it to a specific GPU, you can simply use `--gpus=1` which allocates one GPU to the container, or `--gpus=all` to run with multiple GPUs.

[Launch Llama NIM script](./scripts/run_llama_nim.sh) (click link to view)

In [ ]:
# Launch your NIM server !
!./scripts/run_llama_nim.sh


In [ ]:
# Since we ran the NIM in daemon mode you can continuously check the status of the container using docker logs ... 

# Wait for 60 seconds to let model load start
!sleep 60

# This cell can be run multiple times
!docker logs nim_llama_3.1 2>&1 | tail -n 5

If all is working well you should see messages like this

```
INFO 09-10 13:29:50.476 metrics.py:334] Avg prompt throughput: 0.3 tokens/s, Avg generation throughput: 1.5 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%
```

### Verify NIM has downloaded the weights properly
Here we use a convenience function make sure all the weights are downloaded properly.  We cant move forward until all the weights are downloaded.  For launchpad we have already downloaded the weights, so this step should complete immediately.

In [ ]:
# check for safetensors in nim cache
from utils import safetensors_downloaded
import os
from IPython.display import clear_output
import time

i=1
while(not safetensors_downloaded(directory_path=os.getenv('LOCAL_NIM_CACHE'))) :
    print("Waiting for safetensors to be downloaded in the nim cache - sleeping for 5 seconds")
    time.sleep(5)
    clear_output(wait=True)  # Clear the previous output
    print(f"Iteration {i}")
    i += 1
print("Model Downloaded successfully")


### Verify NIM is running 
Here we will check to see if the NIM server is running.  First we use a curl commnad to show what is returned when querying the server endpoint.  We also include a convenience function to check to make sure the right model is loaded.

In [ ]:
# Quick test to make sure the server is running.
# Check the models that are available
print(f"NIM local api hosted on {LOCAL_MODEL_URL}")
!curl {LOCAL_MODEL_URL}'/v1/models' | jq

In [ ]:
# Check if the model is running
# Define the URL and model ID to check
from utils import check_model_running
url = f'{LOCAL_MODEL_URL}/v1/models'
model_id = os.getenv("NIM_SERVED_MODEL_NAME")

# Check if the model ID exists in the response
while(not check_model_running(url, model_id)):
    print(f"Model ID '{model_id}' not found - Model still loading. Sleeping for 5 seconds.")
    time.sleep(5)
    clear_output(wait=True)  # Clear the previous output

print(f"Model ID '{model_id}' Is running? = {check_model_running(url, model_id)}")



## Testing Llama 8B with NIM

Once the server is up and running, we can query the server using the OpenAI-compatible `/completions` and `/chat/completions/` endpoints that are supplied by the NIM container. <br><br>

**NOTE** : Since we are using the instruction fine tuned version of Llama 8B, only the `/chat/completions` endpoint is valid.  If we had used the Llama 8B languange model that just provides next token prediction, we would use the `/completions` endpoints.
<br><br>
Here, we query the `/chat/completions/` endpoint as an example with the prompt “What are the main branches of the DoD?”:

### NIM Local Inference with Llama - /chat/completions API with curl

In [ ]:
import json
import os

data = {
    "model": os.getenv("NIM_SERVED_MODEL_NAME"),
    "messages": [{"role": "user", "content": "What are the main branches of the DoD?"}],
    "max_tokens": 100,
    "temperature": 0.5,
    "stream": False
}

cmd = f'curl -X POST "{LOCAL_MODEL_URL}/v1/chat/completions" -H "accept: application/json" -H "Content-Type: application/json" -d \'{json.dumps(data)}\''

print(cmd)
!{cmd}|jq

### Same thing but using python requests library

In [ ]:
import requests

url = f"{LOCAL_MODEL_URL}/v1/chat/completions"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json"
}

print("Python Request")
print(f"url = {url}")
print(f"headers = {headers}")
print(f"data = {data}")

response = requests.post(url, headers=headers, json=data)

print(response.json())
print(json.dumps(response.json(), indent=4))

### Local NIM using OpenAI Client

This is where the local NIMs highlight the compatiblity with OpenAI API.  C  Here we use the **OpenAI** client to interact with the local NIM server.

In [ ]:
# Local NIM using OPENAI Client

messages = [{"role": "user", "content": "You are a helpful assistant that provides succinct answers.  What are the main branches of the Department of Defense"}]

# Since NVIDIA NIM for LLMs offers an OpenAI-compatible API,
# we can use the OpenAI Python client
from openai import OpenAI

client = OpenAI(
  base_url = f"{LOCAL_MODEL_URL}/v1/",
  api_key  = "abc123", #needs to be set, but not checked
)

completion = client.chat.completions.create(
  model=os.getenv("NIM_SERVED_MODEL_NAME"),
  messages=messages,
  temperature=0.5,
  top_p=1,
  max_tokens=1024,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="", flush=True)

### LangChain Example - ChatNVIDIA accessing build.nvidia.com models

The [ChatNVIDIA](https://python.langchain.com/docs/integrations/chat/nvidia_ai_endpoints/) class is a LangChain chat model that connects to NVIDIA AI Foundation Endpoints. If you have an NVIDIA_API_KEY environment variable setup , you can access our foundation model endpoints in the same way you would use the ChatOpenAI class.

In [ ]:
# Install some basic langchain dependencies
# !pip install langchain langchain_openai langchain_nvidia_ai_endpoints

In [ ]:
# Validate NVIDIA_API_KEY is set and print error if not.  NVIDIA_API_KEY environment variable required by ChatNVIDIA LLM client from Langchain
try:
    os.environ["NVIDIA_API_KEY"]
except KeyError:
    print("Please set your NVIDIA_API_KEY environment variable")

In [ ]:
# Setup Messages List that will be passed to LangChain LLMs - feel free to modify and experiment!
from langchain.schema import HumanMessage, SystemMessage
question = "What are the responsibilities of the DoD?"
messages = [
    SystemMessage(
        content="You are a helpful assistant that provides succinct answers. "
    ),
    HumanMessage(
        content=f"{question}"
    ),
]

In [ ]:
# This example shows how to create a basic langchain llm with ChatNVIDIA class - here we are hitting the cloud foundation model 
# Documentation => https://python.langchain.com/v0.1/docs/integrations/chat/nvidia_ai_endpoints/

# other examples of models to try : ai-phi-3-mini, ai-llama3-70b, ai-gemma-2b, ai-mixtral-8x22b-instruct.  To get a full list see build.nvidia.com (or just type in an invalid model and review the error message!)

from langchain_nvidia_ai_endpoints import ChatNVIDIA
chat_llm = ChatNVIDIA(
        model=cloud_model,
        temperature=0.5,
        max_tokens=500
    )

response = chat_llm.invoke(messages)
print(response.content)

### LangChain Example - ChatOpenAI and ChatNVIDIA accessing local NIM server
This also works with the ChatOpenAI class.  Here we show how to use the ChatOpenAI class to interact with the local NIM server.

In [ ]:
# ChatOpenAI Example
from langchain_openai import ChatOpenAI


# Setup Messages List that will be passed to LangChain LLMs - feel free to modify and experiment!
from langchain.schema import HumanMessage, SystemMessage
context= ""
question = "What are the responsibilities of the DoD?"
messages = [
    HumanMessage(
        content=f"You are a helpful assistant that provides succinct answers.  {question}"
    ),
]

chat_openai = ChatOpenAI(
    model_name=os.environ["NIM_SERVED_MODEL_NAME"],
    temperature=0.5,
    openai_api_base=f"{LOCAL_MODEL_URL}/v1",
    openai_api_key="not needed"
)

response = chat_openai.invoke(messages)
print(response.content)
#~ 65 tokens/sec on A40

In [ ]:
# ChatNvidia Example
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Setup Messages List that will be passed to LangChain LLMs - feel free to modify and experiment!
from langchain.schema import HumanMessage, SystemMessage
question = "What are the responsibilities of the DoD?"
messages = [
    HumanMessage(
        content=f"You are a helpful assistant that provides succinct answers.  {question}"
    ),
]

chat_nvda = ChatNVIDIA(
    model=os.environ["NIM_SERVED_MODEL_NAME"],
    base_url=f"{LOCAL_MODEL_URL}/v1",
    temperature=0,
    max_tokens=500)

response = chat_nvda.invoke(messages)
print(response.content)
#~ 65 tokens/sec on A40

# VLM Example

VLM (Vision-Language Model) and LLM (Large Language Model) represent distinct, but often complementary, classes of AI systems. Both leverage deep learning, but their architectures and applications vary significantly.

A VLM integrates visual data with textual data to perform tasks that require understanding both modalities. These models are trained to align visual and linguistic representations, enabling cross-modal reasoning.

## VLM vs LLM Comparison
| Feature                  | Large Language Model (LLM)                       | Vision-Language Model (VLM)                   |
|--------------------------|--------------------------------------------------|-----------------------------------------------|
| **Data Type**            | Text data only.                                  | Combines text and visual data.                |
| **Primary Goal**         | Language understanding and generation.           | Cross-modal understanding (vision + language).|
| **Input/Output Modalities** | Text input/output.                             | Text and/or visual input/output.              |
| **Example Models**       | GPT, BERT, LLaMA, Claude.                        | CLIP, llama-vision-instruct , Cosmos-Nemotron-34b                 |
| **Applications**         | Chatbots, summarization, NLU, translation.       | Image captioning, VQA, text-to-image synthesis. |

You can find VLMs on build.nvidia.com as well, here is an example of llama-3.2-11b-vision-instruct!  You can provide it an image, and ask questions.  Here we provide it with our NIM Flowchart and ask about the contents of the image.

In [ ]:
# nvdev/meta/llama-3.2-11b-vision-instruct

import requests, base64
import os
from IPython.display import display, Image

invoke_url = "https://ai.api.nvidia.com/v1/gr/nvdev/meta/llama-3.2-11b-vision-instruct/chat/completions"

image_path = "images/pull_sequence2.png"
with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

assert len(image_b64) < 180_000, \
    "To upload larger images, use the assets API (see docs)"

# Display the image in Jupyter Notebook
print("Displaying the image being processed:")
display(Image(filename=image_path))

headers = {
    "Authorization": f'Bearer {os.environ["NGC_API_KEY"]}',
    "Accept": "application/json"
}

payload = {
    "model": 'nvdev/meta/llama-3.2-11b-vision-instruct',
    "messages": [
        {
        "role": "user", "content": f'What is in this image? <img src="data:image/png;base64,{image_b64}" />'
        }
    ],
    "max_tokens": 512,
    "temperature": 1.00,
    "top_p": 1.00,
    "stream": False
}

# Parse the response
response = requests.post(invoke_url, headers=headers, json=payload)
rv = json.loads(response.content.decode("utf-8"))
print(rv['choices'][0]['message']['content'])


## Conclusion
In this notebook, we showed how to: 

1. Use NVIDIA Foundation Model Endpoints to run inference on NVIDIA hosted models,
2. Deploy an optimized model using NVIDIA NIM for LLMs,
3. Use cURL and `requests` to query the NVIDIA Inference Microservice, and
4. Use the ChatNvidia and ChatOpenAi classes with `langchain` to perform basic inference as a building block that can be used for more complex workflows.

### Appendix

### Using Docker Compose instead of Docker Run
Instead of using `docker run` which often leads to multi-line commands, all configuration settings can be equivalently defined as a YAML file and the inference server can be started using `docker compose`. We show an example YAML file in this section to do so.  

[Docker Compose Yaml Example](docker/docker-compose-example.yaml)

```bash
# Start the server using docker compose, assumes the default yaml file is named docker-compose.yaml
docker compose up nim_llama_8b
```


## Troubleshooting
### Container is already started
If nim_llama is already started, you can stop it with the following convenience script:
```
!bash scripts/stop_containers.sh
```
